# DocMind in the cloud (Kaggle or Colab, free)
Runs the heavy parts off a local machine: ingestion, retrieval ablation, RAGAS with a Groq judge. Everything is resumable.

**Before you run:** (Kaggle) Settings > Internet **On** (needs a phone-verified account) and Add-ons > Secrets > add `GROQ_API_KEY`. (Colab) the key icon > add `GROQ_API_KEY`, enable Notebook access.
Run the cells top to bottom. Keep this notebook **private**: your documents and questions pass through it.

In [ ]:
# 1. Locate the project. Option A: you uploaded DocMind-fixed.zip as a PRIVATE Kaggle Dataset (auto-unzipped under /kaggle/input).
#    Option B (Colab): upload the zip in the Files panel. Option C: set REPO_URL to a git repo you own.
import os, sys, glob, subprocess, zipfile, shutil
from pathlib import Path
REPO_URL = ""   # e.g. "https://github.com/<you>/DocMind.git"   (leave empty to use a zip)
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content") if Path("/content").exists() else Path.cwd()
hits = [Path(p).parents[1] for p in glob.glob("/kaggle/input/**/scripts/cloud_run.py", recursive=True)]
if hits:
    PROJECT = WORK / "DocMind"; shutil.copytree(hits[0], PROJECT, dirs_exist_ok=True)
elif glob.glob(str(WORK / "DocMind*.zip")) or glob.glob("/kaggle/input/**/DocMind*.zip", recursive=True):
    z = (glob.glob(str(WORK / "DocMind*.zip")) + glob.glob("/kaggle/input/**/DocMind*.zip", recursive=True))[0]
    zipfile.ZipFile(z).extractall(WORK / "_unz")
    PROJECT = next(Path(p).parents[1] for p in glob.glob(str(WORK / "_unz/**/scripts/cloud_run.py"), recursive=True))
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK / "DocMind")], check=True); PROJECT = WORK / "DocMind"
else:
    raise SystemExit("Project not found: add the zip as a Kaggle Dataset, upload it, or set REPO_URL.")
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT)); print("project:", PROJECT)

In [ ]:
# 2. Install (about 2 to 4 minutes the first time). Ignore pip's dependency warnings about preinstalled packages.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements-eval.txt"], check=True)
import nltk; nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)

In [ ]:
# 3. Secrets and environment. The key is read from the platform's secret store, never typed into the notebook.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        print("No secret store found: set os.environ['GROQ_API_KEY'] yourself (do not commit this notebook with the key).")
import torch
os.environ["DOCMIND_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
os.environ["DOCMIND_HOME"] = str(Path(PROJECT) / "home")
print("device:", os.environ["DOCMIND_DEVICE"], "| key set:", bool(os.environ.get("GROQ_API_KEY")))

In [ ]:
# 4. Your documents. Add them to the same private Kaggle Dataset (or upload to Colab), then list them here.
import glob
DOCS = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True) + glob.glob("/kaggle/input/**/*.docx", recursive=True)
              + glob.glob("/content/*.pdf") + glob.glob(str(Path(PROJECT) / "data/*.pdf")))
print(DOCS)
assert DOCS, "No documents found. Add PDFs/DOCX to your Kaggle Dataset or upload them."

In [ ]:
# 5a. (Skip if data/testset.json already exists.) Draft questions with the 70B model. Nothing is trusted yet.
#     Download data/testset.draft.json, verify it locally with scripts/verify_testset.py, upload data/testset.json back.
docs_arg = " ".join(f'"{d}"' for d in DOCS)
!python scripts/make_testset.py --docs {docs_arg} --n 60 --unanswerable 12 --draft-only --out data/testset.draft.json

In [ ]:
# 5b. The evidence run. Resumable: if the Groq daily cap is hit it stops cleanly (exit code 2); run this cell again later.
#     --limit 10 is a cheap trial. Remove it for the full run.
!python scripts/cloud_run.py --docs {docs_arg} --testset data/testset.json --chunkers --limit 10

In [ ]:
# 6. Results. Kaggle: the zip appears under Output after "Save Version". Colab: it downloads below.
# Unzip it into the local project folder: reports/ and indexes/docmind.db land in the right places.
import os
print(os.path.getsize("docmind_results.zip") / 1e6, "MB")
print(open("reports/retrieval_latest.md").read() if os.path.exists("reports/retrieval_latest.md") else "no retrieval report")
try:
    from google.colab import files; files.download("docmind_results.zip")
except Exception:
    pass